<a href="https://colab.research.google.com/github/Maxiwel-Cloud/ESG-Analytics-S-P500/blob/main/The_ESG_Streamlit_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit



In [30]:
import subprocess

# Install localtunnel to expose the Streamlit app
subprocess.run(['npm', 'install', '-g', 'localtunnel'], check=True)


CompletedProcess(args=['npm', 'install', '-g', 'localtunnel'], returncode=0)

In [31]:
%%writefile app.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv("preprocessed_content.csv")

# Drop unnecessary column
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

st.title("📊 ESG Performance Dashboard (S&P 500)")

# Sidebar filters
year = st.sidebar.selectbox("Select Year", sorted(df['year'].unique()))
ticker = st.sidebar.selectbox("Select Company", sorted(df['ticker'].unique()))

filtered_df = df[df['year'] == year]

# Average ESG Scores by Year
st.subheader("Average ESG Score by Year")
year_avg = df.groupby('year')['total_score'].mean()

fig1, ax1 = plt.subplots()
year_avg.plot(kind='line', ax=ax1)
st.pyplot(fig1)

# Pillar Comparison
st.subheader("Average ESG Pillar Scores")
pillar_avg = filtered_df[['e_score','s_score','g_score']].mean()

fig2, ax2 = plt.subplots()
pillar_avg.plot(kind='bar', ax=ax2)
st.pyplot(fig2)

# Company Specific View
st.subheader(f"ESG Scores for {ticker} in {year}")
company_data = df[(df['ticker'] == ticker) & (df['year'] == year)]

st.write(company_data[['e_score','s_score','g_score','total_score']])

Overwriting app.py


In [32]:
from pyngrok import ngrok
import subprocess
import time
import threading
from google.colab import userdata

def run_streamlit_if_not_running():
    # Check if a Streamlit process is already running
    try:
        output = subprocess.check_output(['pgrep', '-f', 'streamlit run app.py']).decode().strip()
        if output:
            print("Streamlit app appears to be already running.")
            return
    except subprocess.CalledProcessError:
        # No streamlit process found
        pass

    print("Starting Streamlit app in a background thread...")
    process = subprocess.Popen(["streamlit", "run", "app.py"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    time.sleep(7) # Increased sleep for more robustness
    print("Streamlit app background start initiated.")

# --- Set ngrok authentication token from Colab Secrets ---
# Ensure you have your NGROK_AUTH_TOKEN saved in Colab's 'Secrets' tab (🔑 icon on the left panel).
try:
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
    if NGROK_AUTH_TOKEN:
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        print("ngrok authentication token set from Colab secrets.")
    else:
        print("NGROK_AUTH_TOKEN not found in Colab secrets. Please add it.")
except Exception as e:
    print(f"Error retrieving NGROK_AUTH_TOKEN from Colab secrets: {e}")

print("Attempting to establish ngrok tunnel...")
try:
    tunnel = ngrok.connect(addr=8501)
    print(f"Streamlit app is now accessible at: {tunnel.public_url}")
except Exception as e:
    print(f"Error establishing ngrok tunnel: {e}")
    print("Please ensure:")
    print("1. ngrok executable is in the current directory or PATH.")
    print("2. Your ngrok authentication token is set correctly (preferably via Colab secrets).")
    print("3. The Streamlit app is actually running on port 8501.")

Error retrieving NGROK_AUTH_TOKEN from Colab secrets: Secret NGROK_AUTH_TOKEN does not exist.
Attempting to establish ngrok tunnel...
Streamlit app is now accessible at: https://bloomy-jonnie-bashfully.ngrok-free.dev
